<a href="https://colab.research.google.com/github/dylandlr/Agon/blob/main/agon_optuna_hyperparamater_tuning_a100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Optuna Hyperparameter Sweep

In [1]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 29.9 MB/s eta 0:00:00


In [2]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # let training subprocesses use the whole GPU
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"    # prevent 17.8GiB BFC fragmentation OOM
!nvidia-smi --query-gpu=name,memory.total --format=csv
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert jax.default_backend() == "gpu", "No GPU: Runtime > Change runtime type > A100 GPU, then restart."
if "A100" not in str(jax.devices()[0].device_kind):
    print("Note: not an A100. Everything works, but the time estimates above will be longer.")

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
JAX 0.11.1 devices: [CudaDevice(id=0)]
Note: not an A100. Everything works, but the time estimates above will be longer.


In [3]:
import glob, json, os, shutil, subprocess, sys, zipfile
ZIP_FROM_DRIVE = ""   # e.g. "/content/drive/MyDrive/agon/lifesim_code.zip"
if ZIP_FROM_DRIVE and os.path.exists(ZIP_FROM_DRIVE):
    zip_path = ZIP_FROM_DRIVE
else:
    from google.colab import files
    zip_path = next(iter(files.upload()))
shutil.rmtree("/content/lifesim", ignore_errors=True)
os.makedirs("/content/lifesim")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/lifesim")
if not os.path.exists("/content/lifesim/pyproject.toml"):
    inner = next(os.path.dirname(p) for p in glob.glob("/content/lifesim/*/pyproject.toml"))
    for name in os.listdir(inner):
        shutil.move(os.path.join(inner, name), "/content/lifesim")
os.chdir("/content/lifesim")
!pip install -q -U "flax>=0.12.10" "optax>=0.2.8" pytest
!pip install -q --no-deps -e .
import importlib.metadata as md
v = {p: md.version(p) for p in ("jax", "flax", "optax")}
print(v)
assert tuple(int(x) for x in v["flax"].split(".")[:3]) >= (0, 12, 10), "Flax too old: Runtime > Restart session, rerun this cell."
sys.path.insert(0, "/content/lifesim")
from lifesim.presets import PRESETS, code_version
PRESET = "society30"
print("code version", code_version(), "| preset", PRESET, PRESETS[PRESET])

Saving lifesim_code (47).zip to lifesim_code (47).zip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 38.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lifesim (pyproject.toml) ... done
{'jax': '0.11.1', 'flax': '0.12.10', 'optax': '0.2.8'}
code version f8beb2434918 | preset society30 {'agents': 64, 'founders': 8, 'reproduction': True, 'reward_mode': 'survival', 'grid': 32, 'steps': 128, 'encoder': 'conv', 'slate': 'caveman', 'world': 'living', 'world_model': True, 'episode_years': 25, 'days_per_bio_year': 40.0, 'tick_minutes': 30.0}


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from lifesim.presets import code_version
PRESET = "society30"
ROOT = f"/content/drive/MyDrive/agon/runs/exp_{code_version()}_{PRESET}"
os.makedirs(ROOT, exist_ok=True)
print("experiments saved to", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
experiments saved to /content/drive/MyDrive/agon/runs/exp_d8fb4066274f_society30


In [4]:
import os
import subprocess
import sys
import gc
import optuna

# 1. Inspect how train_phase2.py takes arguments so we know our tuning hooks work
print("=== CHECKING TRAIN_PHASE2.PY ARGUMENTS ===")
help_res = subprocess.run(
    [sys.executable, "/content/lifesim/scripts/train_phase2.py", "--help"],
    capture_output=True, text=True
)
print(help_res.stdout or help_res.stderr)

# 2. Clear any local memory held by the main Colab kernel
gc.collect()

def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    entropy_coef = trial.suggest_float("entropy_coef", 0.01, 0.15)

    trial_out = f"/content/drive/MyDrive/agon/runs/optuna_trials/trial_{trial.number}"
    os.makedirs(trial_out, exist_ok=True)

    cmd = [
        sys.executable, "/content/lifesim/scripts/train_phase2.py",
        "--preset", "society30",
        "--worlds", "1",
        "--ticks", "3000",
        "--log-every", "10",
        "--ckpt-every", "10",
        "--out", trial_out
    ]

    env = os.environ.copy()
    # Let the subprocess use async CUDA malloc WITHOUT artificial 0.3/0.4 caps
    env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    env["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # Forces JAX to allocate/free exact memory on demand
    env.pop("XLA_PYTHON_CLIENT_MEM_FRACTION", None)
    env["LIFESIM_LR"] = str(lr)
    env["LIFESIM_ENTROPY_COEF"] = str(entropy_coef)

    print(f"\n--- Starting Trial {trial.number}: lr={lr:.5f}, entropy_coef={entropy_coef:.4f} ---")
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"GPU failed, falling back to CPU for Trial {trial.number}...")
        env["CUDA_VISIBLE_DEVICES"] = ""
        result = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Trial {trial.number} crashed:\n{result.stderr[-500:]}")
            return 0.0

    best_lifespan = 0.0
    for line in result.stdout.splitlines():
        if "lifespan" in line:
            try:
                parts = line.split()
                idx = parts.index("lifespan")
                val = float(parts[idx + 1])
                if val > best_lifespan:
                    best_lifespan = val
            except (ValueError, IndexError):
                pass

    print(f"Trial {trial.number} Result -> Max Mean Lifespan: {best_lifespan:.1f}")
    return best_lifespan

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=3, timeout=1200)

=== CHECKING TRAIN_PHASE2.PY ARGUMENTS ===


[I 2026-10-21 08:43:16,422] A new study created in memory with name: no-name-939869dd-e2c5-4aeb-813a-6198f05775e6


usage: train_phase2.py [-h] [--agents AGENTS] [--worlds WORLDS]
                       [--ticks TICKS] [--steps STEPS] [--grid GRID]
                       [--scarcity SCARCITY] [--encoder {conv,mlp}]
                       [--gamma GAMMA] [--trait-gamma] [--no-agent-lr]
                       [--seed SEED] [--msg-ent-coef MSG_ENT_COEF]
                       [--speech-attention SPEECH_ATTENTION]
                       [--teach-bonus TEACH_BONUS] [--no-sleep-learning]
                       [--reproduction] [--founders FOUNDERS]
                       [--reward-mode {acts,survival}] [--alpha ALPHA]
                       [--imitation IMITATION]
                       [--imitation-bias {uniform,success,kin}]
                       [--imitate-all] [--log-every LOG_EVERY] [--out OUT]
                       [--ckpt-every CKPT_EVERY] [--resume]
                       [--slate {blank,caveman}] [--saint-rate SAINT_RATE]
                       [--consolidation-ticks CONSOLIDATION_TICKS]
      

[I 2026-10-21 08:50:30,496] Trial 0 finished with value: 0.0 and parameters: {'lr': 0.0002368263634642381, 'entropy_coef': 0.13753450306763387}. Best is trial 0 with value: 0.0.


Trial 0 Result -> Max Mean Lifespan: 0.0

--- Starting Trial 1: lr=0.00025, entropy_coef=0.0720 ---


[W 2026-10-21 08:50:43,152] Trial 1 failed with parameters: {'lr': 0.00024628839761586834, 'entropy_coef': 0.07199598606069667} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_3622/3953351730.py", line 44, in objective
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
  File "/usr/lib/python3.13/subprocess.py", line 556, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
                     ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 1222, in communicate
    stdout, stderr = self._communicate(input, endtime, timeout)
                     ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 2154, in _communicate
    ready = selector.select(timeout)
  File "/usr/

KeyboardInterrupt: 

In [ ]:
!pip install -q optuna
!python scripts/tune.py --out /content/drive/MyDrive/agon/runs/tune_society30 --trials 12 --seeds 2 -- \
    --preset society30 --worlds 8 --ticks 64000 --log-every 25 --agent-chunk 4

[I 2026-10-20 20:27:42,218] A new study created in Journal with name: agon

=== trial 0: lr 3.00e-04, ent_coef 0.0050
  seed 0
    update 25: survival 1.5982  (2475.4 agent-steps/s)
    update 50: survival 1.694  (3178.6 agent-steps/s)
    update 75: survival 1.6346  (3511.1 agent-steps/s)
    update 100: survival 1.2889  (3705.1 agent-steps/s)
    update 125: survival 1.6508  (3831.8 agent-steps/s)
    update 150: survival 1.4023  (3921.7 agent-steps/s)
    update 175: survival 1.6356  (3988.6 agent-steps/s)
    update 200: survival 1.8623  (4040.1 agent-steps/s)
    update 225: survival 1.8682  (4081.1 agent-steps/s)
    update 250: survival 1.8985  (4114.5 agent-steps/s)
    update 275: survival 1.9003  (4142.2 agent-steps/s)
    update 300: survival 2.0791  (4165.5 agent-steps/s)
    update 325: survival 2.0328  (4185.5 agent-steps/s)
    update 350: survival 2.0111  (4202.8 agent-steps/s)
    update 375: survival 2.0578  (4218.0 agent-steps/s)
    update 400: survival 1.994  (4231

In [ ]:
!python scripts/tune.py --out /content/drive/MyDrive/agon/runs/tune_f8beb24 --trials 8 --seeds 2 --tune-sleep -- \
    --preset society30 --worlds 8 --ticks 24000 --log-every 25 --agent-chunk 4

[I 2026-10-21 08:50:46,323] A new study created in Journal with name: agon

=== trial 0: lr 3.00e-04, ent_coef 0.0050, consolidation 8.0, retention 0.100
  seed 0
